# Black-Scholes Comparison

In this notebook, we compare Monte Carlo option prices with the Black-Scholes analytical formula.

For simple European call and put options, Black-Scholes gives a closed-form benchmark. This allows us to check whether the Monte Carlo simulation is producing reasonable results.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.simulation import simulate_gbm_paths
from src.options import price_european_option_mc
from src.black_scholes import black_scholes_call, black_scholes_put

## Parameters

We use the same parameters for both methods.

The Monte Carlo method estimates the option price through simulation, while Black-Scholes computes the analytical benchmark directly.

In [ ]:
S0 = 100
K = 110
r = 0.05
sigma = 0.20
T = 1.0

n_steps = 252
n_paths = 100_000

## Monte Carlo prices

We simulate stock price paths, extract the terminal prices, and estimate European call and put prices with Monte Carlo.

In [ ]:
paths = simulate_gbm_paths(
    S0=S0,
    r=r,
    sigma=sigma,
    T=T,
    n_steps=n_steps,
    n_paths=n_paths,
    seed=42,
)

terminal_prices = paths[:, -1]

mc_call = price_european_option_mc(
    terminal_prices=terminal_prices,
    K=K,
    r=r,
    T=T,
    option_type="call",
)

mc_put = price_european_option_mc(
    terminal_prices=terminal_prices,
    K=K,
    r=r,
    T=T,
    option_type="put",
)

mc_call, mc_put

## Black-Scholes prices

We now compute the analytical Black-Scholes prices for the same European call and put options.

In [ ]:
bs_call = black_scholes_call(
    S0=S0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
)

bs_put = black_scholes_put(
    S0=S0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
)

bs_call, bs_put

## Comparison

We compare the Monte Carlo estimates with the Black-Scholes analytical prices.

The Monte Carlo result should be close to the Black-Scholes value, but not exactly identical because it is based on random simulation.

In [ ]:
comparison = pd.DataFrame(
    {
        "Option": ["Call", "Put"],
        "Monte Carlo Price": [mc_call["price"], mc_put["price"]],
        "Black-Scholes Price": [bs_call, bs_put],
        "Absolute Error": [
            abs(mc_call["price"] - bs_call),
            abs(mc_put["price"] - bs_put),
        ],
        "Standard Error": [
            mc_call["standard_error"],
            mc_put["standard_error"],
        ],
        "95% Confidence Interval": [
            mc_call["confidence_interval"],
            mc_put["confidence_interval"],
        ],
    }
)

comparison

## Interpretation

The Monte Carlo prices are close to the Black-Scholes prices.

This is important because it validates the simulation pipeline:

1. stock paths are simulated correctly;
2. payoffs are computed correctly;
3. discounting is applied correctly;
4. the Monte Carlo estimate is consistent with the analytical benchmark.

The small difference between the two methods is due to Monte Carlo sampling error.